In [1]:
# !pip install ta-lib
# !pip install gdown
# !pip install requests
# !pip install numpy
# !pip install pandas

In [2]:
# !pip install -r requirements_dev.txt       
# !pip install pyotp
# !pip install logzero
# !pip install websocket-client    

In [3]:
# !pip uninstall pycrypto
# !pip install pycryptodome    

In [4]:
import pandas as pd
# import numpy as np
import requests
from datetime import datetime
import socket
import uuid
# import http.client
import time
import requests # type: ignore
# import mimetypes
import json
import talib
# import gdown
import http
import ssl
import os

In [5]:
# !pip install python-dotenv

In [6]:
from SmartApi import SmartConnect #or from SmartApi.smartConnect import SmartConnect
import pyotp
from logzero import logger

In [7]:
from dotenv import load_dotenv
load_dotenv()

True

In [8]:
# Static values
user_type = "USER"
source_id = "WEB"   
api_key = os.environ["ANG_ONE_KEY"]  
client_code = os.environ["CLIENTCODE"]
password = os.environ["PASSWORD"]
window = 965

bot_token =  os.environ["BOT_TOKEN"]
test_mode = os.environ["TEST_MODE"].lower() == 'true'


# todays_date = datetime.today().strftime("%Y-%m-%d")
todays_date = (datetime.today() - pd.DateOffset(days=0)).strftime("%Y-%m-%d")
# window_date = (datetime.today() - pd.DateOffset(days=window)).strftime("%Y-%m-%d")
window_date = datetime(2025,1,1).strftime("%Y-%m-%d")
# '2025-01-01'  # Future date to include all data

In [9]:
local_ip = socket.gethostbyname(socket.gethostname())
smartApi = SmartConnect(api_key)

try:
    token = os.environ["TOTP_TOKEN"]
    totp = pyotp.TOTP(token).now()
except Exception as e:
    logger.error("Invalid Token: The provided token is not valid.")
    raise e


# Get Public IP
public_ip = requests.get('https://api.ipify.org').text

# Get MAC Address
mac_address = ':'.join(['{:02x}'.format((uuid.getnode() >> ele) & 0xff)
                        for ele in range(0,8*6,8)][::-1])

# Change clientcode, password, totp
payload = '''{\n\"clientcode\":\"'''+str(client_code)+'''\"
         ,\n\"password\":\"'''+str(password)+'''\"\n
		,\n\"totp\":\"'''+str(totp)+'''\"\n
    ,\n\"state\":\"Active\"\n}'''

headers = {
    'Content-Type': 'application/json',
    'Accept': 'application/json',
    "X-UserType": user_type,
    "X-SourceID": source_id,
    "X-ClientLocalIP": local_ip,
    "X-ClientPublicIP": public_ip,
    "X-MACAddress": mac_address,
    'X-PrivateKey': api_key 
}


context = ssl._create_unverified_context()

conn = http.client.HTTPSConnection(
    "apiconnect.angelone.in", context=context
    )

conn.request("POST", "/rest/auth/angelbroking/user/v1/loginByPassword", payload, headers)

res = conn.getresponse()
data = res.read()
data = data.decode("utf-8")

[I 251119 17:54:31 smartConnect:121] in pool


In [10]:
temp = json.loads(data)
jwtToken = temp["data"]["jwtToken"]
# print(jwtToken)


local_ip = socket.gethostbyname(socket.gethostname())
public_ip = requests.get('https://api.ipify.org').text
mac_address = ':'.join(['{:02x}'.format((uuid.getnode() >> ele) & 0xff)
                        for ele in range(0,8*6,8)][::-1])
authToken = f'Bearer {jwtToken}'


headers = {
    'Content-Type': 'application/json',
    'Accept': 'application/json',
    "X-UserType": user_type,
    "X-SourceID": source_id,
    "X-ClientLocalIP": local_ip,
    "X-ClientPublicIP": public_ip,
    "X-MACAddress": mac_address,
    'X-PrivateKey': api_key,
    'Authorization': authToken ,
}


In [11]:
# shareable_link = 'https://drive.google.com/file/d/1PdYMxjWQ4tBJp4Mmp1LjLOR2H2vkZ6on/view?usp=sharing'

# Extract the file ID
# file_id = shareable_link.split('/d/')[1].split('/view')[0]

# Construct the download URL
# download_url = f'https://drive.google.com/uc?id={file_id}'


# # Download the file using gdown
# output_file = 'Nifty500-token.csv'

# stock_symbols_df = pd.read_csv(output_file)
# stock_symbols_df["token"] = stock_symbols_df["token"].fillna(891)
# stock_symbols_df["token"] = stock_symbols_df["token"].astype(int)
# stocks_to_consider = 'PO1_Stocks.csv'

# stock_lists = pd.read_csv(stocks_to_consider)
# main_df = pd.merge(stock_lists[['rsi','symbol','win_ratio','priority']], stock_symbols_df[['Symbol','token']], left_on ='symbol' , right_on='Symbol', how='inner')

# main_df.to_csv('Main_df.csv', index=False)

In [12]:

# Download the file using gdown
output_file = 'Nifty500-token.csv'
stock_symbols_df = pd.read_csv(output_file)
stock_symbols_df["token"] = stock_symbols_df["token"].fillna(891)
stock_symbols_df["token"] = stock_symbols_df["token"].astype(int)
# stocks_to_consider = 'Stocks.csv'

# stock_lists = pd.read_csv(stocks_to_consider)
# full_main_df = pd.merge(stock_lists[['rsi','symbol','win_ratio','priority']], stock_symbols_df[['Symbol','token']], left_on ='symbol' , right_on='Symbol', how='inner')


# # full_main_df.to_csv('Full_Main_df.csv', index=False)

main_df = stock_symbols_df.copy()

# full_main_df = pd.read_csv('Full_Main_df.csv')

In [21]:
# Step 2: Function to fetch daily candle data from API
def fetch_candle_data(symbol,interval='ONE_DAY'):

    all_data = []
    start_date = window_date
    print(symbol, start_date)
    end_date = todays_date

    while start_date < end_date:
        time.sleep(0.4)  # To avoid hitting API rate limits
        chunk_end = (pd.to_datetime(start_date) + pd.DateOffset(days=30)).strftime("%Y-%m-%d")
        print(f"Fetching: {start_date}  →  {chunk_end}")
        
        payload = '''{\r\n     \"exchange\": \"NSE\",\r\n
          \"symboltoken\": \"'''+str(symbol)+'''\",\r\n     \"interval\": \"'''+str(interval)+'''\",\r\n
          \"fromdate\": \"'''+str(start_date)+''' 09:15\",\r\n     \"todate\": \"'''+str(chunk_end)+''' 16:30\"\r\n}
    '''

        conn = http.client.HTTPSConnection("apiconnect.angelone.in", context=context)
        conn.request("POST", "/rest/secure/angelbroking/historical/v1/getCandleData", payload, headers)
        res = conn.getresponse()
        data = res.read()
        data = data.decode("utf-8")
        json_data = json.loads(data)
        json_data = json_data['data']
       
        
        if not json_data:
            print(f"No data returned for {start_date} to {chunk_end}.")
            raise json_data['message']
            break
        all_data.extend(json_data)
        

        start_date = (pd.to_datetime(chunk_end)  + pd.DateOffset(days=1)).strftime("%Y-%m-%d")

    # if all_data:
        # print(all_data)
    return all_data

    

    

In [14]:
# daily_json_data = fetch_candle_data(395,'FIVE_MINUTE')
# df = pd.DataFrame(daily_json_data, columns=["Date", "Open", "High", "Low", "Close", "Volume"])
        
#         # break

#         # Convert 'Date' column to datetime
# df['Date'] = pd.to_datetime(df['Date'])

#         # Sort data by date in ascending order
# df = df.sort_values('Date').reset_index(drop=True)

# df['RSI_14'] = talib.RSI(df['Close'], timeperiod=14)

# print(df)

In [ ]:
# Prepare output DataFrame
error_data = []

final_df = pd.DataFrame()
print(main_df.columns.tolist())

# Step 5: Process each stock
#iterate only first 5 rows from 2nd row ingnore 1st

# for _, row in main_df.head(50).iterrows():
for _, row in main_df.iloc[101:150].iterrows():

    time.sleep(0.4)

    name = row['Symbol']
    token = row['token']
    
    try:   
        # Fetch daily data
        daily_json_data = fetch_candle_data(token,'FIVE_MINUTE')
        
        if daily_json_data == None:
            error_data.append({
               'Name': name,
               'Token': token,
               'reasone': 'Error while fetching data'
             })    
            continue
        
        # logger.info(f"Processing {name} with token {token} and RSI {rsi}")
        
        df = pd.DataFrame(daily_json_data, columns=["Date", "Open", "High", "Low", "Close", "Volume"])
        
        # break

        # Convert 'Date' column to datetime
        df['Date'] = pd.to_datetime(df['Date'])

        # Sort data by date in ascending order
        df = df.sort_values('Date').reset_index(drop=True)

        df['RSI_14'] = talib.RSI(df['Close'], timeperiod=14)
        # keep 2 numbers after decimal  
        df['RSI_14'] = df['RSI_14'].round(2)
        df.set_index('Date', inplace=True)
        df['Token'] = token
        df['Symbol'] = name
        
        # final_df = pd.concat([final_df, df])
        
        # last_2_rsi_daily = df['RSI_14'].dropna().tail(2).values      
        last_dats = daily_json_data[-1][0].split('T')[0]
        
        df.to_csv(f'5min data/Final_5min_RSI_{name}_from_{window_date}_to_{last_dats}.csv')
        

    except Exception as e:
        error_data.append({
            'Name': name,
            'Token': token,
            'reasone': str(e)
        })
        print(f"Error processing {name}: {e}")

['Symbol', 'NAME OF COMPANY', 'token']
14894 2025-01-01
Fetching: 2025-01-01  →  2025-01-31
Fetching: 2025-02-01  →  2025-03-03
Fetching: 2025-03-04  →  2025-04-03
Fetching: 2025-04-04  →  2025-05-04
Fetching: 2025-05-05  →  2025-06-04
Fetching: 2025-06-05  →  2025-07-05
Fetching: 2025-07-06  →  2025-08-05
Fetching: 2025-08-06  →  2025-09-05
Fetching: 2025-09-06  →  2025-10-06
Fetching: 2025-10-07  →  2025-11-06
Fetching: 2025-11-07  →  2025-12-07
21174 2025-01-01
Fetching: 2025-01-01  →  2025-01-31
Fetching: 2025-02-01  →  2025-03-03
Fetching: 2025-03-04  →  2025-04-03
Fetching: 2025-04-04  →  2025-05-04
Fetching: 2025-05-05  →  2025-06-04
Fetching: 2025-06-05  →  2025-07-05
Fetching: 2025-07-06  →  2025-08-05
Fetching: 2025-08-06  →  2025-09-05
Fetching: 2025-09-06  →  2025-10-06
Fetching: 2025-10-07  →  2025-11-06
Fetching: 2025-11-07  →  2025-12-07
13305 2025-01-01
Fetching: 2025-01-01  →  2025-01-31
Fetching: 2025-02-01  →  2025-03-03
Fetching: 2025-03-04  →  2025-04-03
Fetching: 

In [16]:
# print(main_df)


In [17]:
# import requests

# BOT_TOKEN = bot_token

# TEST_ID = "529251493"

# if test_mode:
#     CHAT_ID = TEST_ID
# else:
#     CHAT_ID = "-1003139839259"
    
# def format_whatsapp_report(data ,name):
    
    
#     if len(data) == 0:
#         # lines.append( "\n🔹 <b>No Stocks</b>" )
#         # lines.append( "<b>: No Stocks</b>" )
#         lines = [f"📊 <b>{name} : 0 Stocks </b>"]
#     else:
#         for index,item in enumerate(data):
#             lines = [f"📊 <b>{name}</b>"]
#             lines.append(
#                 f"\n🔹 <b>{index+1} {item['Name']}</b>"
#                 f"\n   <b>Todays RSI: </b> {item['Daily_RSI']:.2f}"
#                 f"\n   <b>Yesterdays RSI: </b> {item['yesterday_RSI']:.2f}"
#                 f"\n   <b>Standard RSI: </b> {item['Setup_RSI']:.2f}"
#                 # f"\n   High Priority: {'✅' if item['High Priority'] else '❌'}"
#             )      
           
#     # return urllib.parse.quote_plus( "\n".join(lines) )
#     return "\n".join(lines) 

# def format_whatsapp_error(data ,name):
    
#     lines = [f"📊 <b>{name}</b>"]
    
#     if len(data) == 0:
#         lines.append( "\n🔹 <b>No Stocks</b>" )
#     else:
#         for index,item in enumerate(data):
#             lines.append(
#                 f"\n🔹 <b>{index+1} {item['Name']}</b>"
#                 f"\n   <b>Token: </b> {item['Token']}"
#                 f"\n   <b>Standard RSI: </b> {item['Setup_RSI']:.2f}"
#                 f"\n   <b>Reasone: </b> {item['reasone']}"
#                 # f"\n   High Priority: {'✅' if item['High Priority'] else '❌'}"
#             )      
           
#     # return urllib.parse.quote_plus( "\n".join(lines) )
#     return "\n".join(lines)

# def format_whatsapp_40_report(data ,name):
    
#     # lines = [f"📊 <b>{name}</b>"]
    
#     if len(data) == 0:
#         lines = [f"📊 <b>{name} : 0 Stocks </b>"]
#         # lines.append( "<b>: No Stocks</b>" )
#     else:
#         for index,item in enumerate(data):
#             lines = [f"📊 <b>{name}</b>"]
#             lines.append(
#                 f"\n🔹 <b>{index+1} {item['Name']}</b>"
#                 f"\n   <b>Current Month RSI: </b>{item['Monthly_RSI']:.2f}"
#                 f"\n   <b>Last Month RSI: </b>{item['last_Month_RSI']:.2f}"
#                 f"\n   <b>Priority: </b>{item['Priority']}"
#             )      
           
#     # return urllib.parse.quote_plus( "\n".join(lines) )
#     return "\n".join(lines) 

In [18]:
# # Telegram Message trigger logic
# msg = f"📊 <b>Daily Report: {todays_date}</b>"
# requests.get(f"https://api.telegram.org/bot{BOT_TOKEN}/sendMessage",
#              params={"chat_id": CHAT_ID, "text": msg, "parse_mode": "HTML"})

# msg_p1 = format_whatsapp_report(priority_data,'Priority Stocks')
# requests.get(f"https://api.telegram.org/bot{BOT_TOKEN}/sendMessage",
#              params={"chat_id": CHAT_ID, "text": msg_p1, "parse_mode": "HTML"})


# msg_p2 = format_whatsapp_report(priority_watch_data,'Priority Stocks to be tracked')
# requests.get(f"https://api.telegram.org/bot{BOT_TOKEN}/sendMessage",
#              params={"chat_id": CHAT_ID, "text": msg_p2, "parse_mode": "HTML"})

# msg_t = format_whatsapp_report(output_data ,'Treading Stocks')
# requests.get(f"https://api.telegram.org/bot{BOT_TOKEN}/sendMessage",
#              params={"chat_id": CHAT_ID, "text": msg_t, "parse_mode": "HTML"})

# msg_p0 = format_whatsapp_report(priority0_data,'Least Priority Stocks')
# requests.get(f"https://api.telegram.org/bot{BOT_TOKEN}/sendMessage",
#             params={"chat_id": CHAT_ID, "text": msg_p0, "parse_mode": "HTML"})


In [19]:

# if len(error_data) > 0:
#     error_msg = format_whatsapp_error(error_data,'Error Stocks')
#     requests.get(f"https://api.telegram.org/bot{BOT_TOKEN}/sendMessage",
#                 params={"chat_id": TEST_ID, "text": error_msg, "parse_mode": "HTML"})

# # if len(rsi_40_cross_data) > 0:
# msg_40 = format_whatsapp_40_report(rsi_40_cross_data,'RSI 40 Crossover Stocks')
# requests.get(f"https://api.telegram.org/bot{BOT_TOKEN}/sendMessage",
#                             params={"chat_id": TEST_ID, "text": msg_40, "parse_mode": "HTML"})